# Cellular RNA simulation

Generates synthetic datasets and fits continuous and categorical BMI models.
The manuscript uses 20 conditions × 10 datasets × 2 models = 400 assessments.

1. Run the dependency cell in Colab and set `OUTPUT_BASE` to a new output directory.
2. Use four runtimes with `SHARD_ID` set to 0, 1, 2 and 3. Keep the other settings identical.
3. Set `RUN_SIMULATION=True` and execute the cells in order. Completed tasks are reused.
4. Once all four shards finish, set `RUN_SIMULATION=False`, `AGGREGATE=True` and run the notebook in one runtime. Set `DOWNLOAD_REPORTS=True` to download its summaries.

Keep `REPLICATES_TO_RUN=10` for the manuscript design. `PLANNED_REPLICATES=50`
is part of the original experiment contract; the remaining repetitions were not run.


In [ ]:
# 0. Dependencies: run before importing scientific packages
import importlib.util, subprocess, sys
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "pymc==5.28.5", "arviz==0.23.4", "nutpie==0.16.11",
        "pandas>=2.2,<3", "scipy>=1.12,<2", "h5netcdf>=1.3", "h5py>=3.9"])

In [ ]:
# Run controls
from pathlib import Path
RUN_SIMULATION = False
AGGREGATE = False
DOWNLOAD_REPORTS = False
N_SHARDS = 4
SHARD_ID = 0                      # One runtime per shard: 0, 1, 2, 3.
SUITE = 'focused'
PLANNED_REPLICATES = 50           # Retained in the experiment contract.
REPLICATES_TO_RUN = 10            # Manuscript: 400 model assessments.
MAX_NEW_TASKS_PER_LAUNCH = None
REPLICATES = PLANNED_REPLICATES
SEED = 20260910
ALLOW_PUBLICATION_LONG = True
SAVE_TRACES = True
OUTPUT_BASE = Path('/content/drive/MyDrive/BHM_simulation_results_v5_release')

GEOMETRIES = ['generic_K4', 'generic_K8', 'generic_K16', 'generic_K8_wide']
APPLICATION_INVENTORIES = {'immune_coarse':'', 'immune_fine':'', 'nonimmune':''}
USE_APPLICATION_ANCHORS = False
# Optional inventories require global_feature and global_celltype columns.
# Enabling them changes the simulation design and experiment identity.
if USE_APPLICATION_ANCHORS:
    GEOMETRIES += ['immune_coarse','immune_fine','nonimmune']


In [ ]:
# 2. Shared definitions and analysis contract
import os, math, json, hashlib, inspect, time, gc, traceback, uuid
import numpy as np
import pandas as pd
import xarray as xr
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import importlib.metadata
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist, norm, t as student_t
from IPython.display import display
from dataclasses import dataclass, asdict, replace
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
assert 0 <= SHARD_ID < N_SHARDS and N_SHARDS >= 1
assert REPLICATES >= 1
VERSION = "bhm-simulation-v5.0-focused"
PRIORS = dict(baseline_sd=.5, celltype_bmi_sd=.2, feature_bmi_sd=.3,
    celltype_covariate_sd=.25, feature_covariate_sd=.25, patient_global_sd=.5,
    log_sigma_location=math.log(.65), log_sigma_location_sd=.4,
    celltype_log_sigma_sd=.25, feature_log_sigma_sd=.5, nu_rate=.1)
PROFILES = {"publication": dict(tune=2000, draws=4000, chains=4, target_accept=.99),
            "publication_long": dict(tune=2000, draws=8000, chains=4, target_accept=.99)}
Q_LEVEL=.05
ROPE_GRID=(.05,.10,.15,.20)
PRIMARY_ROPE=.10
MC_ALPHA=.025
ROPE_INDICATOR_ESS_MIN=400.
ROPE_LOCAL_NULL_MCSE_MAX=.005
BFDR_MCSE_MAX=.0025
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()
def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def rng_for(*parts):
    return np.random.default_rng(int(digest([SEED,*parts])[:16],16))
def atomic_json(value,path):
    path=Path(path); tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,default=str)); os.replace(tmp,path)
def save_trace(idata,path):
    path=Path(path)
    if path.exists(): raise FileExistsError(path)
    tmp=path.with_name(path.stem+'.'+uuid.uuid4().hex+'.tmp.nc')
    idata.to_netcdf(tmp); os.replace(tmp,path)
def resolve_geometry(name):
    if name.startswith('generic_K'):
        k=int(name.split('K')[-1]); counts=[6]*k
        labels=[f'generic_ct_{i:02d}' for i in range(k)]
        source=None
    else:
        path=Path(APPLICATION_INVENTORIES.get(name,''))
        if not path.is_file():
            raise ValueError(f'Supply audited feature_inventory.csv for {name}')
        inv=pd.read_csv(path)
        if not {'global_feature','global_celltype'} <= set(inv):
            raise ValueError('Inventory needs global_feature and global_celltype')
        if inv.global_feature.duplicated().any(): raise ValueError('Duplicate features')
        groups=inv.groupby('global_celltype',sort=True).size()
        labels=groups.index.astype(str).tolist(); counts=groups.astype(int).tolist()
        source=dict(path=str(path),sha256=file_hash(path))
    if not counts or min(counts)<1: raise ValueError('Empty geometry')
    return dict(name=name,labels=labels,counts=counts,source=source)
def resolve_geometry_v5(name):
    if name == 'generic_K8_wide':
        return dict(name=name,labels=[f'generic_ct_{i:02d}' for i in range(8)],counts=[24]*8,source=None)
    resolved = resolve_geometry(name)
    if resolved['source']:
        inventory = pd.read_csv(resolved['source']['path'])
        for column in ('global_feature','global_celltype'):
            if inventory[column].isna().any() or inventory[column].astype(str).str.strip().eq('').any():
                raise ValueError(f'Missing or blank {column} in {name}')
        if len(inventory) != sum(resolved['counts']):
            raise ValueError('Inventory grouping lost features')
    return resolved
GEOMETRY={g:resolve_geometry_v5(g) for g in GEOMETRIES}
@dataclass(frozen=True)
class Scenario:
    name: str
    shape: str='linear'
    counts: tuple=(52,57,18)
    effect: float=.20
    active: float=.25
    error: str='student_t'
    rho: float=0.
    df: float=8.
    confounding: float=.2
    bmi_prior_multiplier: float=1.
    pooling: bool=True
    patient_re: bool=True
    likelihood: str='student_t'
    dgp_patient: str='restricted'

def scenarios_for(suite):
    if suite=='core':
        return [Scenario(f'{shape}_effect_{e}',shape=shape,effect=e)
                for shape in ('linear','threshold','plateau') for e in (.10,.20,.50)] + [Scenario('null',shape='null',effect=0)]
    if suite=='robustness':
        return [Scenario('reference'),Scenario('gaussian_errors',error='normal'),Scenario('correlated',rho=.6),
            Scenario('t3',df=3),Scenario('contaminated',error='contaminated'),
            Scenario('heteroskedastic',error='heteroskedastic'),
            Scenario('unrestricted_patient',dgp_patient='iid'),
            Scenario('confounding',confounding=.65),
            Scenario('obese10',counts=(52,57,10)),Scenario('obese40',counts=(52,57,40)),
            Scenario('balanced',counts=(42,42,43)),Scenario('smaller',counts=(26,29,9)),
            Scenario('larger',counts=(104,114,36)),
            Scenario('sparse',active=.05),Scenario('dense',active=.50)]
    if suite=='prior_sensitivity':
        return [Scenario(f'bmi_prior_{m}',bmi_prior_multiplier=m) for m in (.5,1.,2.)]
    if suite=='ablation':
        return [Scenario('full'),Scenario('no_BMI_pooling',pooling=False),
            Scenario('no_patient',patient_re=False),Scenario('gaussian',likelihood='normal')]
    if suite=='sbc': return [Scenario('prior_joint')]
    raise ValueError(suite)
SCENARIOS=[] # populated by focused_scenarios below

In [ ]:
# NetCDF metadata encoding and checkpoint verification
import os, json, uuid, hashlib, gc, re, traceback
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import xarray as xr
import arviz as az

IO_REPAIR_VERSION = 'v5-netcdf-metadata-and-text-repair-3'
# This reference remains reachable if saving fails, allowing retry in this kernel.
_UNSAVED_SIMULATION_TRACE = globals().get('_UNSAVED_SIMULATION_TRACE')

def _metadata_json_default(value):
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, Path): return str(value)
    if isinstance(value, bytes): return {'bytes_hex': value.hex()}
    raise TypeError(f'Unsupported metadata type: {type(value).__name__}')

def _netcdf_attrs(attrs):
    out = {}
    for key, value in attrs.items():
        if not isinstance(key, str): raise TypeError('Metadata keys must be strings')
        if isinstance(value, (bool, np.bool_)):
            out[key] = int(value)
        elif isinstance(value, (str, int, float, np.integer, np.floating)):
            out[key] = value
        else:
            # Includes nested sampler settings, None and containers. Preserve contents.
            out[key] = json.dumps(value, default=_metadata_json_default, sort_keys=True)
    return out

def _netcdf_copy(idata):
    groups = {}
    for name in idata.groups():
        ds = getattr(idata, name).copy(deep=False)
        ds.attrs = _netcdf_attrs(ds.attrs)
        for variable in ds.variables:
            ds[variable].attrs = _netcdf_attrs(ds[variable].attrs)
            if ds[variable].dtype.kind == 'O':
                values = ds[variable].values
                if not all(v is None or isinstance(v, str) for v in values.flat):
                    raise TypeError(f'Unsupported object array: {name}/{variable}')
                missing = [i for i, v in enumerate(values.flat) if v is None]
                encoded = np.asarray(['' if v is None else v for v in values.flat], dtype=str).reshape(values.shape)
                attrs = dict(ds[variable].attrs)
                if missing:
                    attrs['netcdf_missing_string_flat_indices'] = json.dumps(missing)
                ds[variable] = xr.Variable(ds[variable].dims, encoded, attrs=attrs)
        groups[name] = ds
    return az.InferenceData(attrs=_netcdf_attrs(idata.attrs), **groups)

def _checkpoint_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def _checkpoint_json(value, path):
    path = Path(path)
    tmp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=_metadata_json_default))
    os.replace(tmp, path)

def _verify_checkpoint(original, path):
    check = az.from_netcdf(path, engine='h5netcdf')
    try:
        if set(check.groups()) != set(original.groups()):
            raise ValueError('Checkpoint group mismatch')
        canonical = _netcdf_copy(original)
        for name in original.groups():
            # Compares every coordinate and data value; metadata is encoded separately.
            xr.testing.assert_equal(getattr(canonical, name), getattr(check, name))
        for key in ('config_hash', 'task_id', 'profile'):
            if check.attrs.get(key) != original.attrs.get(key):
                raise ValueError(f'Checkpoint identity mismatch: {key}')
    finally:
        check.close()

def save_trace(idata, path):
    global _UNSAVED_SIMULATION_TRACE
    path = Path(path)
    if path.exists(): raise FileExistsError(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    _UNSAVED_SIMULATION_TRACE = {'idata': idata, 'path': str(path)}
    safe = _netcdf_copy(idata)
    tmp = path.with_name(path.stem + '.' + uuid.uuid4().hex + '.tmp.nc')
    safe.to_netcdf(tmp, engine='h5netcdf')
    _verify_checkpoint(idata, tmp)
    if path.exists(): raise FileExistsError(path)
    os.replace(tmp, path)
    receipt = {
        'io_repair_version': IO_REPAIR_VERSION,
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'config_hash': idata.attrs.get('config_hash'),
        'task_id': idata.attrs.get('task_id'), 'profile': idata.attrs.get('profile'),
        'sha256': _checkpoint_sha(path), 'size_bytes': path.stat().st_size,
        'groups': list(idata.groups()), 'all_data_and_coordinates_verified': True,
        'metadata_encoding': 'Nested/other non-scalar attributes encoded as JSON text; booleans as integers',
        'text_encoding': 'Object text normalized to strings; None positions preserved in variable attributes. Numerical arrays unchanged.',
    }
    _checkpoint_json(receipt, path.with_suffix('.checkpoint.json'))
    _UNSAVED_SIMULATION_TRACE = None
    print('SAVED + VERIFIED:', path, flush=True)

def checkpoint_io_preflight(root, shard_id):
    """Tiny synthetic I/O round trip on the actual target filesystem; no MCMC."""
    dest = Path(root) / '_io_checks'
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f'shard_{shard_id}_{uuid.uuid4().hex}.nc'
    coords = {'chain': np.arange(4), 'draw': np.arange(3)}
    test = az.InferenceData(
        attrs={'config_hash': 'io-check', 'task_id': 'io-check', 'profile': 'io-check',
               'sample_stats': {'inference_library': 'nutpie',
                                'inference_library_version': '0.16.11',
                                'inference_library_settings': '{"sampler":"nuts"}'}},
        posterior=xr.Dataset({'test_effect': (('chain', 'draw'), np.arange(12.).reshape(4, 3))}, coords=coords),
        sample_stats=xr.Dataset({'diverging': (('chain', 'draw'), np.zeros((4, 3), dtype=bool)),
            'divergence_message': (('chain', 'draw'), np.array([[None, '', 'test']]*4, dtype=object))}, coords=coords),
    )
    test.posterior.attrs['nested'] = {'check': [True, None, 3]}
    test.posterior.test_effect.attrs['nested'] = {'source': 'preflight'}
    save_trace(test, path)
    print('PASS: nested metadata and posterior arrays save/reload correctly. No model was sampled.', flush=True)
    return path

def simulation_progress(tasks, title, root, config_hash):
    from collections import Counter
    counts = Counter(); statuses = Counter()
    for args in tasks:
        tid = task_id(*args); folder = Path(root) / 'tasks' / tid
        result = folder / 'result.json'
        traces = [folder / (p + '.nc') for p in ('publication', 'publication_long')]
        if result.exists():
            record = json.loads(result.read_text())
            if record.get('config_hash') != config_hash or record.get('task_id') != tid:
                raise ValueError(f'Incompatible result: {result}')
            counts['finished_assessments'] += 1; statuses[record['status']] += 1
        elif any(p.exists() for p in traces):
            counts['saved_fit_without_final_result'] += 1
        elif (folder / 'failure.json').exists(): counts['error_without_saved_fit'] += 1
        elif any(folder.glob('*.tmp.nc')): counts['temporary_file_only'] += 1
        else: counts['no_checkpoint'] += 1
        if (folder / 'failure.json').exists(): counts['error_logs_including_historical'] += 1
    print('\n', title, '| Snapshot:', datetime.now(timezone.utc).isoformat())
    print('Experiment:', root)
    print('Finished assessments:', counts['finished_assessments'], '/', len(tasks))
    for name in ['saved_fit_without_final_result', 'error_without_saved_fit', 'temporary_file_only', 'no_checkpoint', 'error_logs_including_historical']:
        print(name + ':', counts[name])
    print('Finished statuses:', dict(statuses))
    return dict(counts)

def run_shard_safely():
    """Unexpected implementation errors stop the shard. Recorded withholds remain valid outcomes."""
    if not SAVE_TRACES: raise ValueError('This repaired runner requires SAVE_TRACES=True')
    checkpoint_io_preflight(OUTPUT_ROOT, SHARD_ID)
    for i, args in enumerate(MY_TASKS, 1):
        print(f'Shard {SHARD_ID}: {i}/{len(MY_TASKS)}', flush=True)
        try:
            run_task(*args)
        except Exception:
            tid = task_id(*args); folder = OUTPUT_ROOT / 'tasks' / tid
            folder.mkdir(parents=True, exist_ok=True)
            error = traceback.format_exc()
            record = dict(config_hash=CONFIG_HASH, task_id=tid, error=error,
                          io_repair_version=IO_REPAIR_VERSION,
                          created_utc=datetime.now(timezone.utc).isoformat())
            # Preserve previous errors, plus the conventional latest-error file.
            try:
                _checkpoint_json(record, folder / ('failure_' + uuid.uuid4().hex + '.json'))
                _checkpoint_json(record, folder / 'failure.json')
            except Exception:
                print('Could not save error log:', traceback.format_exc(), flush=True)
            print(error, flush=True)
            print('STOPPED. No next task was launched. Keep this runtime connected.', flush=True)
            raise
        if i == 1 or i % 10 == 0:
            simulation_progress(MY_TASKS, f'SHARD {SHARD_ID}', OUTPUT_ROOT, CONFIG_HASH)
    simulation_progress(MY_TASKS, f'SHARD {SHARD_ID} FINAL', OUTPUT_ROOT, CONFIG_HASH)

def recover_live_simulation_traces(base):
    """Best effort only: scan reachable InferenceData objects in THIS Python process."""
    base = Path(base)
    if not base.is_dir(): raise FileNotFoundError('Mount the original Drive first: ' + str(base))
    experiments = {}
    for manifest in base.glob('*/experiment.json'):
        record = json.loads(manifest.read_text())
        key = record.get('config_hash')
        if key:
            if key in experiments: raise ValueError('Ambiguous experiment identity: ' + key)
            experiments[key] = (manifest.parent, record)
    # Do not call gc.collect(): unreachable objects may still contain salvageable arrays.
    # Inspect concrete types, without dereferencing weak proxies or invoking an
    # unrelated object's __class__ property. Expired proxies can raise ReferenceError
    # during isinstance(); their real, live referents are enumerated separately.
    candidates = [obj for obj in gc.get_objects()
                  if issubclass(type(obj), az.InferenceData)]
    recovered = 0; matching = 0
    for candidate in candidates:
        attrs = candidate.attrs
        if attrs.get('config_hash') not in experiments: continue
        if not hasattr(candidate, 'posterior') or not hasattr(candidate, 'sample_stats'): continue
        root, manifest = experiments[attrs['config_hash']]
        tid = attrs.get('task_id', ''); profile = attrs.get('profile', '')
        if not re.fullmatch(r'[A-Za-z0-9_.-]+', tid) or tid in {'.', '..'}: continue
        if profile not in {'publication', 'publication_long'}: continue
        expected = manifest['contract']['profiles'][profile]
        if candidate.posterior.sizes.get('chain') != expected['chains'] or candidate.posterior.sizes.get('draw') != expected['draws']:
            print('Incomplete in-memory trace:', tid); continue
        if 'feature_bmi' not in candidate.posterior: continue
        matching += 1; path = root / 'tasks' / tid / (profile + '.nc')
        if path.exists():
            _verify_checkpoint(candidate, path)
            print('Already saved:', path); continue
        save_trace(candidate, path); recovered += 1
    print('Matching complete in-memory traces:', matching)
    print('Newly recovered trace files:', recovered)
    if not matching:
        print('No matching trace remains in this runtime. Error logs cannot reconstruct posterior draws.')
    return recovered


In [ ]:
# 3. DGP, design and structural truth
def patient_frame(s,r,g):
    rng=rng_for('patients',g,r,s.counts,s.confounding)
    bmi=np.concatenate([np.clip(rng.normal(22.2,1.4,s.counts[0]),18.5,24.99),
        np.clip(rng.normal(27.4,1.3,s.counts[1]),25,29.99),
        np.clip(rng.normal(34,3,s.counts[2]),30,45)])
    group=np.repeat(np.arange(3),s.counts); order=rng.permutation(len(bmi))
    bmi=bmi[order]; group=group[order]; sd=bmi.std(ddof=1)
    z=(bmi-bmi.mean())/sd
    age=s.confounding*z+np.sqrt(1-s.confounding**2)*rng.normal(size=len(z))
    sex=rng.binomial(1,1/(1+np.exp(.15-.3*z)))-.5
    return pd.DataFrame(dict(bmi=bmi,bmi_z=z,group=group,age_z=age,sex_centered=sex)),sd
def design(p,kind):
    e=p[['bmi_z']].to_numpy() if kind=='continuous' else np.column_stack([p.group==1,p.group==2]).astype(float)
    cov=p[['age_z','sex_centered']].to_numpy()
    return np.column_stack([np.ones(len(p)),e,cov]),e,cov
def residual_basis(x):
    u,sv,_=np.linalg.svd(x,full_matrices=True)
    rank=int((sv>max(x.shape)*np.finfo(float).eps*sv.max()).sum())
    if rank>=len(x): raise ValueError('No residual patient dimensions')
    b=u[:,rank:]*np.sqrt(len(x)/(len(x)-rank))
    assert np.max(np.abs(x.T@b))<1e-9
    assert np.isclose(np.mean(np.sum(b*b,axis=1)),1)
    return b
def simulate(s,r,g):
    geom=GEOMETRY[g]; fmap=np.repeat(np.arange(len(geom['counts'])),geom['counts'])
    k=len(geom['counts']); f=len(fmap)
    p,bmi_sd=patient_frame(s,r,g); n=len(p)
    rng=rng_for('latent',g,r)
    active=np.zeros(f,bool); active[rng.permutation(f)[:round(s.active*f)]]=True
    magnitude=rng.lognormal(-.02,.2,f)
    signs=rng.choice([-1.,1.],k)[fmap]
    beta=s.effect*magnitude*signs*active
    if s.shape=='null': basis=np.zeros(n)
    elif s.shape=='linear': basis=p.bmi_z.to_numpy()
    elif s.shape=='threshold': basis=(p.group==2).to_numpy(float)
    elif s.shape=='plateau': basis=(p.group!=0).to_numpy(float)
    else: raise ValueError(s.shape)
    baseline=rng.normal(0,.25,f)
    gamma=rng.normal(0,.12,(2,k))[:,fmap]+rng.normal(0,.08,(2,f))
    mean=baseline[None,:]+basis[:,None]*beta[None,:]+p[['age_z','sex_centered']].to_numpy()@gamma
    union=np.column_stack([design(p,'categorical')[0],p.bmi_z])
    b=residual_basis(union)
    patient=.35*(b@rng.normal(size=b.shape[1])) if s.dgp_patient=='restricted' else rng.normal(0,.35,n)
    scales=np.exp(np.log(.65)+rng.normal(0,.15,k)[fmap]+rng.normal(0,.25,f))
    # Within-cell dependence via a Gaussian copula, maintaining marginal t errors.
    e=rng.normal(size=(n,f)); shared=rng.normal(size=(n,k))[:,fmap]
    z=np.sqrt(1-s.rho)*e+np.sqrt(s.rho)*shared
    nu=np.maximum(2.2,s.df*np.exp(np.linspace(-.15,.15,k)))
    if s.error=='normal': errors=z*scales
    else:
        errors=student_t.ppf(np.clip(norm.cdf(z),1e-12,1-1e-12),nu[fmap])*scales
        if s.error=='contaminated': errors*=np.where(rng.random((n,f))<.03,5.,1.)
        elif s.error=='heteroskedastic': errors*=np.exp(.25*p.bmi_z.to_numpy())[:,None]
        elif s.error!='student_t': raise ValueError(s.error)
    y=mean+patient[:,None]+errors
    return dict(p=p,y=y,structural_mean=mean,fmap=fmap,bmi_sd=bmi_sd,scales=scales,nu=nu)
def targets(d,kind):
    x,e,c=design(d['p'],kind)
    projection=np.linalg.lstsq(x,d['structural_mean'],rcond=None)[0]
    if kind=='continuous': return {'bmi':projection[1]}
    return {'overweight_vs_normal':projection[1], 'obese_vs_normal':projection[2],
            'obese_vs_overweight':projection[2]-projection[1]}

In [ ]:
# 4. Model matched to normalized-state v2.2/v2.2c
def build_model(d,kind,s):
    p=d['p']; y=d['y']; n,f=y.shape; fmap=d['fmap']; k=int(fmap.max()+1)
    x,e,c=design(p,kind); b=residual_basis(x)
    exposure=['bmi'] if kind=='continuous' else ['overweight_vs_normal','obese_vs_normal']
    coords=dict(patient=np.arange(n),feature=[f'f{j:04d}' for j in range(f)],
        celltype=np.arange(k),exposure=exposure,covariate=['age_z','sex_centered'],
        patient_re_dimension=np.arange(b.shape[1]))
    with pm.Model(coords=coords) as m:
        base=pm.Normal('feature_baseline',0,PRIORS['baseline_sd'],dims='feature')
        if s.pooling:
            ct=pm.Normal('celltype_bmi',0,PRIORS['celltype_bmi_sd']*s.bmi_prior_multiplier,dims=('exposure','celltype'))
            tau=pm.HalfNormal('feature_sigma_bmi',PRIORS['feature_bmi_sd']*s.bmi_prior_multiplier,dims=('exposure','celltype'))
            raw=pm.Normal('feature_bmi_raw',0,1,dims=('exposure','feature'))
            beta=pm.Deterministic('feature_bmi',ct[:,fmap]+raw*tau[:,fmap],dims=('exposure','feature'))
        else:
            beta=pm.Normal('feature_bmi',0,PRIORS['feature_bmi_sd']*s.bmi_prior_multiplier,dims=('exposure','feature'))
        ct_cov=pm.Normal('celltype_covariate_slope',0,PRIORS['celltype_covariate_sd'],dims=('covariate','celltype'))
        tau_cov=pm.HalfNormal('feature_sigma_covariate',PRIORS['feature_covariate_sd'],dims=('covariate','celltype'))
        raw_cov=pm.Normal('feature_covariate_raw',0,1,dims=('covariate','feature'))
        gamma=pm.Deterministic('feature_covariate_slope',ct_cov[:,fmap]+raw_cov*tau_cov[:,fmap],dims=('covariate','feature'))
        if s.patient_re:
            ps=pm.HalfNormal('patient_global_sigma',PRIORS['patient_global_sd'])
            coeff=pm.Normal('patient_global_coefficient',0,ps,dims='patient_re_dimension')
            patient=pm.Deterministic('patient_global_effect',pt.dot(b,coeff),dims='patient')
        else: patient=pt.zeros(n)
        loc=pm.Normal('log_sigma_location',PRIORS['log_sigma_location'],PRIORS['log_sigma_location_sd'])
        ct_sd=pm.HalfNormal('log_sigma_celltype_location_sd',PRIORS['celltype_log_sigma_sd'])
        ct_raw=pm.Normal('log_sigma_celltype_raw',0,1,dims='celltype')
        ct_loc=pm.Deterministic('log_sigma_celltype_location',loc+ct_sd*ct_raw,dims='celltype')
        fs=pm.HalfNormal('log_sigma_feature_scale',PRIORS['feature_log_sigma_sd'],dims='celltype')
        fr=pm.Normal('log_sigma_raw',0,1,dims='feature')
        sigma=pm.Deterministic('sigma_feature',pt.exp(ct_loc[fmap]+fs[fmap]*fr),dims='feature')
        mu=base[None,:]+pt.dot(e,beta)+pt.dot(c,gamma)+patient[:,None]
        if s.likelihood=='student_t':
            nu=pm.Deterministic('nu',pm.Exponential('nu_minus_two',PRIORS['nu_rate'],dims='celltype')+2,dims='celltype')
            pm.StudentT('y_obs',nu=nu[fmap][None,:],mu=mu,sigma=sigma[None,:],observed=y,dims=('patient','feature'))
        else:
            pm.Normal('y_obs',mu=mu,sigma=sigma[None,:],observed=y,dims=('patient','feature'))
    return m
def extract_effects(idata,kind):
    a=idata.posterior.feature_bmi
    if kind=='continuous': return {'bmi':a.sel(exposure='bmi',drop=True)}
    ow=a.sel(exposure='overweight_vs_normal',drop=True)
    ob=a.sel(exposure='obese_vs_normal',drop=True)
    return {'overweight_vs_normal':ow,'obese_vs_normal':ob,'obese_vs_overweight':ob-ow}
def sample(m,profile,seed):
    with m:
        return pm.sample(**PROFILES[profile],cores=4,nuts_sampler='nutpie',
            nuts_sampler_kwargs={'maxdepth':10},random_seed=seed,progressbar=True)
def diagnostics(idata):
    table=az.summary(idata,kind='diagnostics',round_to='none')
    stats=idata.sample_stats
    missing=[v for v in ['diverging','energy','maxdepth_reached'] if v not in stats]
    if missing: raise RuntimeError(f'Missing sampler statistics: {missing}')
    bfmi=np.asarray(az.bfmi(idata))
    finite=bool(np.isfinite(table[['r_hat','ess_bulk','ess_tail']]).all().all() and np.isfinite(bfmi).all())
    div=int(stats.diverging.sum()); depth=float(stats.maxdepth_reached.mean())
    geometry=bool(div==0 and np.isfinite(bfmi).all() and bfmi.min()>=.3 and depth<=.01)
    conv=bool(finite and table.r_hat.max()<=1.01 and table.ess_bulk.min()>=400 and table.ess_tail.min()>=400)
    return dict(geometry_pass=geometry,convergence_pass=conv,divergences=div,
        maxdepth_fraction=depth,min_bfmi=float(bfmi.min()),max_rhat=float(table.r_hat.max()),
        min_ess_bulk=float(table.ess_bulk.min()),min_ess_tail=float(table.ess_tail.min()))

In [ ]:
def _az_da_stat(da, statistic: str, method: str):
    dataset = xr.Dataset({"metric": da.astype(float)})
    function = az.ess if statistic == "ess" else az.mcse
    return function(dataset, var_names=["metric"], method=method)["metric"]


def effective_sample_boundary_guard(mean, n_eff, alpha=MC_ALPHA):
    """Conservative Monte Carlo boundary guard, not an exact data CI."""
    n_eff = max(1.0, float(n_eff))
    mean = float(np.clip(mean, 0, 1))
    x = mean * n_eff
    lower = 0.0 if x <= 0 else float(beta_dist.ppf(alpha, x, n_eff - x + 1))
    upper = 1.0 if x >= n_eff else float(
        beta_dist.ppf(1 - alpha, x + 1, n_eff - x)
    )
    return lower, upper


def rope_indicator_metrics(beta, effect_dim: str, id_column: str, rope: float):
    beta = beta.transpose("chain", "draw", effect_dim)
    indicator = (np.abs(beta) <= float(rope)).astype(np.float32).rename(
        "rope_indicator"
    )
    probability = indicator.mean(("chain", "draw")).to_numpy()
    ess_raw = _az_da_stat(indicator, "ess", "bulk").to_numpy()
    mcse_raw = _az_da_stat(indicator, "mcse", "mean").to_numpy()
    beta_bulk = _az_da_stat(beta, "ess", "bulk").to_numpy()
    beta_tail = _az_da_stat(beta, "ess", "tail").to_numpy()
    n_total = beta.sizes["chain"] * beta.sizes["draw"]
    fallback = np.minimum(beta_bulk, beta_tail)
    ess = np.where(np.isfinite(ess_raw) & (ess_raw > 0), ess_raw, fallback)
    ess = np.minimum(ess, n_total)
    z_value = norm.ppf(1 - MC_ALPHA)
    rows = []
    for j, name in enumerate(beta.coords[effect_dim].to_numpy()):
        lower, upper = effective_sample_boundary_guard(probability[j], ess[j])
        raw = float(mcse_raw[j]) if np.isfinite(mcse_raw[j]) else 0.0
        guarded = max(
            raw,
            (upper - probability[j]) / z_value,
            (probability[j] - lower) / z_value,
        )
        rows.append({
            id_column: str(name),
            "local_null_probability": float(probability[j]),
            "indicator_ess": float(ess[j]),
            "indicator_mcse_raw": raw,
            "indicator_mcse_guarded": float(guarded),
            "local_null_mc_lower": lower,
            "local_null_mc_upper": upper,
        })
    return indicator, pd.DataFrame(rows)


def deterministic_bfdr(local_null, identifiers, q=Q_LEVEL):
    values = np.asarray(local_null, float)
    identifiers = np.asarray(identifiers, str)
    if not len(values) or not np.isfinite(values).all():
        raise ValueError("Invalid local-null vector")
    order = np.lexsort((identifiers, values))
    cumulative = np.cumsum(values[order]) / np.arange(1, len(values) + 1)
    accepted = np.flatnonzero(cumulative <= q)
    k = int(accepted[-1] + 1) if len(accepted) else 0
    selected = np.zeros(len(values), dtype=bool)
    selected[order[:k]] = True
    return selected, order, cumulative, k


def bounded_series_mc(series, fallback_ess, alpha=MC_ALPHA):
    mean = float(series.mean())
    n_total = series.sizes["chain"] * series.sizes["draw"]
    ess = float(np.asarray(_az_da_stat(series.rename("metric"), "ess", "bulk")))
    mcse = float(np.asarray(_az_da_stat(series.rename("metric"), "mcse", "mean")))
    if not np.isfinite(ess) or ess <= 0:
        ess = float(fallback_ess)
    ess = min(ess, n_total)
    if not np.isfinite(mcse):
        mcse = 0.0
    guard_low, guard_high = effective_sample_boundary_guard(mean, ess, alpha)
    z_value = norm.ppf(1 - alpha)
    lower = max(0.0, min(mean - z_value * mcse, guard_low))
    upper = min(1.0, max(mean + z_value * mcse, guard_high))
    guarded_mcse = max(
        mcse, (upper - mean) / z_value, (mean - lower) / z_value
    )
    return {
        "mean": mean,
        "ess": ess,
        "mcse_raw": mcse,
        "mcse_guarded": guarded_mcse,
        "mc_lower": lower,
        "mc_upper": upper,
    }


def selection_sets(mask, identifiers, cluster_ids):
    return (
        frozenset(np.asarray(identifiers, str)[mask]),
        frozenset(np.asarray(cluster_ids, str)[mask]),
    )


def assess_rope_bfdr(
    beta, effect_dim, id_column, identifiers, cluster_ids, rope, q=Q_LEVEL
):
    if beta.sizes["chain"] < 4:
        raise ValueError("Reveal requires at least four chains")
    indicator, metrics = rope_indicator_metrics(beta, effect_dim, id_column, rope)
    trace_ids = np.asarray(indicator.coords[effect_dim].to_numpy(), str)
    identifiers = np.asarray(identifiers, str)
    raw_cluster_ids = np.asarray(cluster_ids, dtype=object)
    if pd.isna(raw_cluster_ids).any():
        raise ValueError("Missing correlation-cluster ID")
    cluster_ids = raw_cluster_ids.astype(str)
    if not np.array_equal(trace_ids, identifiers):
        raise ValueError("Effect order differs from posterior coordinates")
    if (
        np.any(np.char.strip(cluster_ids) == "")
        or np.any(np.char.lower(cluster_ids) == "nan")
    ):
        raise ValueError("Empty/invalid correlation-cluster ID")

    probability = metrics["local_null_probability"].to_numpy()
    mask, order, cumulative, k = deterministic_bfdr(probability, identifiers, q)
    next_index = None if k == len(probability) else int(order[k])
    critical = mask | (probability <= 0.10)
    if next_index is not None:
        critical[next_index] = True
    critical_metrics = metrics.loc[critical]
    indicator_precision_pass = bool(
        (critical_metrics["indicator_ess"] >= ROPE_INDICATOR_ESS_MIN).all()
        and (
            critical_metrics["indicator_mcse_guarded"]
            <= ROPE_LOCAL_NULL_MCSE_MAX
        ).all()
    )
    if k:
        selected_series = indicator.isel(
            {effect_dim: np.flatnonzero(mask)}
        ).mean(effect_dim)
        selected_mc = bounded_series_mc(
            selected_series, critical_metrics["indicator_ess"].min()
        )
        selected_bound_pass = bool(
            selected_mc["mc_upper"] <= q
            and selected_mc["mcse_guarded"] <= BFDR_MCSE_MAX
        )
    else:
        selected_mc = {
            "mean": 0.0, "ess": None, "mcse_raw": 0.0,
            "mcse_guarded": 0.0, "mc_lower": 0.0, "mc_upper": 0.0,
        }
        selected_bound_pass = True
    if next_index is not None:
        prefix = order[:k + 1]
        next_series = indicator.isel({effect_dim: prefix}).mean(effect_dim)
        next_mc = bounded_series_mc(
            next_series, metrics.iloc[prefix]["indicator_ess"].min()
        )
        next_bound_pass = bool(
            next_mc["mc_lower"] > q
            and next_mc["mcse_guarded"] <= BFDR_MCSE_MAX
        )
    else:
        next_mc = None
        next_bound_pass = True

    full_identifiers, full_clusters = selection_sets(mask, identifiers, cluster_ids)
    loo_rows = []
    for omitted in range(indicator.sizes["chain"]):
        keep = [j for j in range(indicator.sizes["chain"]) if j != omitted]
        loo_probability = indicator.isel(chain=keep).mean(
            ("chain", "draw")
        ).to_numpy()
        loo_mask, _, _, _ = deterministic_bfdr(loo_probability, identifiers, q)
        identifier_set, cluster_set = selection_sets(
            loo_mask, identifiers, cluster_ids
        )
        loo_rows.append({
            "omitted_chain_position": omitted,
            "identifier_set_identical": identifier_set == full_identifiers,
            "cluster_set_identical": cluster_set == full_clusters,
        })
    loo = pd.DataFrame(loo_rows)
    loo_pass = bool(loo[
        ["identifier_set_identical", "cluster_set_identical"]
    ].to_numpy().all())
    ranks = np.empty(len(probability), int)
    ranks[order] = np.arange(1, len(probability) + 1)
    cumulative_by_effect = np.full(len(probability), np.nan)
    cumulative_by_effect[order] = cumulative
    metrics["bfdr_rank"] = ranks
    metrics["cumulative_bfdr_at_rank"] = cumulative_by_effect
    metrics["bfdr_selected"] = mask
    return {
        "pass": bool(
            indicator_precision_pass and selected_bound_pass
            and next_bound_pass and loo_pass
        ),
        "rope": float(rope),
        "n_selected": k,
        "indicator_precision_pass": indicator_precision_pass,
        "selected_mc": selected_mc,
        "selected_bound_pass": selected_bound_pass,
        "next_prefix_mc": next_mc,
        "next_bound_pass": next_bound_pass,
        "loo_pass": loo_pass,
        "effect_metrics": metrics,
        "loo": loo,
        "minimum_critical_indicator_ess": float(
            critical_metrics["indicator_ess"].min()
        ),
        "maximum_critical_indicator_mcse": float(
            critical_metrics["indicator_mcse_guarded"].max()
        ),
    }

In [ ]:
# 5. Assessment helpers
def assess_all(effects):
    report={}; masks={}; allpass=True
    blocks=[]
    for name,a in effects.items():
        ids=a.feature.values.astype(str)
        for rope in ROPE_GRID:
            v=assess_rope_bfdr(a,'feature','id',ids,ids,rope)
            report[f'{name}|{rope}']={k:v[k] for k in ['pass','n_selected','indicator_precision_pass','selected_bound_pass','next_bound_pass','loo_pass','minimum_critical_indicator_ess','maximum_critical_indicator_mcse']}
            allpass &= v['pass']
            if rope==PRIMARY_ROPE: masks[name]=v['effect_metrics'].bfdr_selected.to_numpy(bool)
        blocks.append(a.assign_coords(feature=[name+'||'+str(f) for f in a.feature.values]))
    joint_masks={}
    if len(blocks)>1:
        a=xr.concat(blocks,dim='feature'); ids=a.feature.values.astype(str)
        for rope in ROPE_GRID:
            v=assess_rope_bfdr(a,'feature','id',ids,ids,rope)
            report[f'joint|{rope}']={k:v[k] for k in ['pass','n_selected','indicator_precision_pass','selected_bound_pass','next_bound_pass','loo_pass','minimum_critical_indicator_ess','maximum_critical_indicator_mcse']}
            allpass &= v['pass']
            if rope==PRIMARY_ROPE:
                selected=v['effect_metrics'].bfdr_selected.to_numpy(bool)
                offset=0
                for name,arr in effects.items():
                    joint_masks[name]=selected[offset:offset+arr.sizes['feature']];offset+=arr.sizes['feature']
    return bool(allpass),report,masks,joint_masks
def recovery(a,truth,mask):
    values=a.transpose('chain','draw','feature').values.reshape(-1,len(truth))
    estimate=values.mean(axis=0)
    hdi=az.hdi(xr.Dataset({'effect':a}),hdi_prob=.95)['effect'].transpose('feature','hdi').values
    exact=np.isclose(truth,0,atol=1e-10,rtol=0); practical=np.abs(truth)>PRIMARY_ROPE
    selected=mask.astype(bool); count=int(selected.sum()); nonzero=selected&~exact
    return dict(discoveries=count,coverage=float(np.mean((hdi[:,0]<=truth)&(truth<=hdi[:,1]))),
        bias=float(np.mean(estimate-truth)),rmse=float(np.sqrt(np.mean((estimate-truth)**2))),
        mean_interval_width=float(np.mean(hdi[:,1]-hdi[:,0])),
        practical_power=float(np.mean(selected[practical])) if practical.any() else None,
        exact_null_fpr=float(np.mean(selected[exact])) if exact.any() else None,
        classical_fdp=float((selected&exact).sum()/max(1,count)),
        practical_fdp=float((selected&~practical).sum()/max(1,count)),
        type_s=float(np.mean(np.sign(estimate[nonzero])!=np.sign(truth[nonzero]))) if nonzero.any() else None,
        type_m=float(np.mean(np.abs(estimate[nonzero]/truth[nonzero]))) if nonzero.any() else None)

In [ ]:
# 6. Estimand-matched OLS benchmark
def ols_benchmark(d,kind):
    x,e,c=design(d['p'],kind); y=d['y']; n,f=y.shape
    rank=np.linalg.matrix_rank(x); coef=np.linalg.lstsq(x,y,rcond=None)[0]
    var=np.sum((y-x@coef)**2,axis=0)/(n-rank); inv=np.linalg.inv(x.T@x)
    vectors={}
    if kind=='continuous': vectors['bmi']=np.eye(x.shape[1])[1]
    else:
        vectors['overweight_vs_normal']=np.eye(x.shape[1])[1]
        vectors['obese_vs_normal']=np.eye(x.shape[1])[2]
        vectors['obese_vs_overweight']=vectors['obese_vs_normal']-vectors['overweight_vs_normal']
    truths=targets(d,kind); out=[]
    for name,v in vectors.items():
        est=v@coef; se=np.sqrt(var*(v@inv@v)); df=n-rank
        p=2*student_t.sf(np.abs(est/se),df); order=np.argsort(p,kind='stable')
        ok=np.flatnonzero(p[order]<=Q_LEVEL*np.arange(1,f+1)/f)
        mask=np.zeros(f,bool)
        if len(ok): mask[order[:ok[-1]+1]]=True
        truth=truths[name]; exact=np.isclose(truth,0,atol=1e-10,rtol=0)
        practical=np.abs(truth)>PRIMARY_ROPE; width=student_t.ppf(.975,df)*se
        out.append(dict(contrast=name,rule='OLS_BH_exact_zero',discoveries=int(mask.sum()),
            coverage=float(np.mean((est-width<=truth)&(truth<=est+width))),
            bias=float(np.mean(est-truth)),rmse=float(np.sqrt(np.mean((est-truth)**2))),
            practical_power=float(mask[practical].mean()) if practical.any() else None,
            classical_fdp=float((mask&exact).sum()/max(1,mask.sum())),
            practical_fdp=float((mask&~practical).sum()/max(1,mask.sum()))))
    return out

In [ ]:
def export_simulation_figures(records, table, destination):
    """Only measured simulation outcomes. Every display names its conditioning denominator."""
    import re
    destination=Path(destination); out=destination/'figures';out.mkdir(exist_ok=True)
    files=[]
    def save(fig,stem):
        stem=re.sub(r'[^A-Za-z0-9_]+','_',stem)
        for ext in ('svg','tiff'):
            p=out/(stem+'.'+ext)
            fig.savefig(p,dpi=300,bbox_inches='tight',**({'pil_kwargs':{'compression':'tiff_lzw'}} if ext=='tiff' else {}))
            files.append(dict(path=str(p),sha256=file_hash(p),size_bytes=p.stat().st_size))
        plt.close(fig)
    colors={'completed':'#397b63','withheld_precision':'#d39b3d','withheld_convergence':'#9573ab',
            'withheld_geometry':'#bd5456','unfinished':'#dadde2'}
    with plt.rc_context({'font.size':16,'axes.titlesize':19,'axes.labelsize':17,
                         'xtick.labelsize':14,'ytick.labelsize':14,'legend.fontsize':13}):
        cases=design_cases(GEOMETRIES,SCENARIOS)
        for kind in ('continuous','categorical'):
            fig,ax=plt.subplots(figsize=(13,max(5,.52*len(cases))),constrained_layout=True)
            labels=[f'{g}: {s.name}' for g,s in cases];left=np.zeros(len(cases))
            for state,color in colors.items():
                n=[]
                for g,s in cases:
                    matched=[r for r in records if r['geometry']==g and r['scenario']==s.name and r['kind']==kind]
                    n.append(PLANNED_REPLICATES-len(matched) if state=='unfinished' else sum(r['status']==state for r in matched))
                ax.barh(np.arange(len(cases)),n,left=left,label=state.replace('_',' '),color=color);left+=n
            ax.set(yticks=np.arange(len(cases)),yticklabels=labels,xlabel='Datasets out of planned repetitions',
                   title=f'{kind.capitalize()} model: completion and decision status')
            ax.invert_yaxis();ax.legend(loc='upper center',bbox_to_anchor=(.5,1.20),ncol=3,frameon=False)
            save(fig,'status_'+kind)
        if not table.empty:
            est=table[table.rule=='Bayes_estimation']
            for (geometry,kind,contrast),group in est.groupby(['geometry','kind','contrast']):
                for metric,label,reference in [('coverage','Mean pathway coverage',.95),
                        ('bias','Mean signed estimation error',0),('rmse','Mean dataset RMSE',None),
                        ('mean_interval_width','Mean 95% interval width',None)]:
                    if metric+'_mean' not in group:continue
                    fig,ax=plt.subplots(figsize=(11,max(4,.6*len(group))),constrained_layout=True)
                    for j,(_,row) in enumerate(group.iterrows()):
                        value=row.get(metric+'_mean');se=row.get(metric+'_mcse')
                        if pd.isna(value):continue
                        ax.plot(value,j,'o',color='#315b80')
                        if pd.notna(se):ax.errorbar(value,j,xerr=1.96*se,fmt='none',capsize=3,color='#315b80')
                    labs=[f"{row.scenario} (n={int(row.get(metric+'_n',0))})" for _,row in group.iterrows()]
                    ax.set(yticks=np.arange(len(group)),yticklabels=labs,xlabel=label+'; bars = ±1.96 Monte Carlo SE',
                        title=f'{geometry}: {kind}\n{contrast.replace("_"," ")} — converged fits')
                    if reference is not None:ax.axvline(reference,color='#a54442',ls='--',lw=1.5)
                    if metric=='coverage':ax.set_xlim(0,1.03)
                    save(fig,f'{metric}_{geometry}_{kind}_{contrast}')
            sel=table[table.rule.str.contains('ROPE_BFDR',na=False)]
            for (geometry,kind,contrast,rule),group in sel.groupby(['geometry','kind','contrast','rule']):
                for metric,label,reference in [('practical_fdp','Mean practical false-discovery proportion',Q_LEVEL),
                        ('practical_power','Mean practical power',None)]:
                    if metric+'_mean' not in group:continue
                    fig,ax=plt.subplots(figsize=(11,max(4,.6*len(group))),constrained_layout=True)
                    for j,(_,row) in enumerate(group.iterrows()):
                        value=row.get(metric+'_mean');se=row.get(metric+'_mcse')
                        if pd.isna(value):continue
                        ax.plot(value,j,'o',color='#397b63')
                        if pd.notna(se):ax.errorbar(value,j,xerr=1.96*se,fmt='none',capsize=3,color='#397b63')
                    labs=[f"{row.scenario} (n={int(row.get(metric+'_n',0))})" for _,row in group.iterrows()]
                    ax.set(yticks=np.arange(len(group)),yticklabels=labs,xlabel=label+'; ±1.96 Monte Carlo SE',
                        title=f'{geometry}: {kind}, {contrast.replace("_"," ")}\nSelection-released datasets only; {rule.replace("_"," ")}')
                    if reference is not None:ax.axvline(reference,color='#a54442',ls='--')
                    save(fig,f'{metric}_{geometry}_{kind}_{contrast}_{rule}')
        timing=[]
        for r in records:
            for h in r['profile_history']:
                if h.get('sample_seconds') is not None:
                    timing.append(dict(geometry=r['geometry'],kind=r['kind'],profile=h['profile'],minutes=h['sample_seconds']/60))
        if timing:
            t=pd.DataFrame(timing);t.to_csv(destination/'sampling_timings.csv',index=False)
            groups=list(t.groupby(['geometry','kind','profile']))
            fig,ax=plt.subplots(figsize=(11,max(4,.7*len(groups))),constrained_layout=True)
            for j,(key,g) in enumerate(groups):ax.plot(g.minutes,np.full(len(g),j),'o',alpha=.6,color='#785b88')
            ax.set(yticks=np.arange(len(groups)),yticklabels=[' / '.join(k) for k,g in groups],
                   xlabel='Measured sampling minutes per fit',title='Computational cost on this runtime')
            save(fig,'sampling_runtime')
    atomic_json(dict(config_hash=CONFIG_HASH,format=['svg','tiff'],dpi=300,files=files),out/'figure_manifest.json')
    (out/'FIGURE_NOTES.md').write_text('Status bars include all planned datasets. Unfinished includes unlaunched tasks and interrupted/software-error attempts; use state.json and failure.json to distinguish. Estimation plots condition on converged fits. Selection plots condition on all selection gates passing. Withheld selections never become zero discoveries. Error bars are approximate Monte Carlo intervals across datasets, not Bayesian intervals for one dataset. No error bar is drawn for one repetition. A missing selection figure means no eligible summary, not zero power. Signed bias may cancel across positive and negative truths; interpret with RMSE and absolute error tables. No SBC calibration figure is produced by this operating-characteristic runner.\n')
    return files



# Simulation assessment and scheduling
from datetime import datetime, timezone

def focused_scenarios():
    # A restricted design: targeted stresses at the middle geometry, not a full cross-product.
    return [Scenario('null', shape='null', effect=0),
            Scenario('linear_0.10', effect=.10), Scenario('linear_0.20', effect=.20),
            Scenario('linear_0.50', effect=.50),
            Scenario('threshold_0.20', shape='threshold', effect=.20),
            Scenario('plateau_0.20', shape='plateau', effect=.20),
            Scenario('patients_64', counts=(26,29,9)),
            Scenario('patients_254', counts=(104,114,36)),
            Scenario('obese_10', counts=(52,57,10)),
            Scenario('correlated_0.20', rho=.6),
            Scenario('contaminated_0.20', error='contaminated'),
            Scenario('heavy_tails', df=3), Scenario('sparse', active=.05),
            Scenario('unrestricted_patient', dgp_patient='iid')]

def design_cases(geometries, scenarios):
    cases=[]
    for g in geometries:
        for s in scenarios:
            if g=='generic_K8' or s.name in ('null','linear_0.20'):
                cases.append((g,s))
    return cases

def task_id(g,s,r,kind):
    return f'{g}__{s.name}__rep{r:04d}__{kind}'

def task_schedule(geometries, scenarios, replicates, shards, owner):
    # Replicate outermost visits all settings early. A complete model pair has one owner.
    all_tasks=[]; own=[]; pair_index=0
    for r in range(replicates):
        for g,s in design_cases(geometries,scenarios):
            pair=[(g,s,r,kind) for kind in ('continuous','categorical')]
            all_tasks.extend(pair)
            if (pair_index + r) % shards == owner: own.extend(pair)
            pair_index+=1
    return all_tasks,own

def estimation_metrics(effects,truth,fmap):
    summaries=[]; features=[]
    for name,a in effects.items():
        vals=a.transpose('chain','draw','feature').values.reshape(-1,len(truth[name]))
        mean=vals.mean(axis=0)
        hdi=az.hdi(xr.Dataset({'a':a}),hdi_prob=.95).a.transpose('feature','hdi').values
        error=mean-truth[name]; covered=(hdi[:,0]<=truth[name])&(truth[name]<=hdi[:,1])
        active=np.abs(truth[name])>PRIMARY_ROPE
        summaries.append(dict(contrast=name,rule='Bayes_estimation',bias=float(error.mean()),
            mean_absolute_error=float(np.abs(error).mean()),mse=float(np.mean(error**2)),
            rmse=float(np.sqrt(np.mean(error**2))),coverage=float(covered.mean()),
            mean_interval_width=float(np.mean(hdi[:,1]-hdi[:,0])),
            active_coverage=float(covered[active].mean()) if active.any() else None,
            practical_null_coverage=float(covered[~active].mean()) if (~active).any() else None))
        for j,f in enumerate(a.feature.values):
            features.append(dict(contrast=name,feature=str(f),celltype=int(fmap[j]),
                truth=float(truth[name][j]),posterior_mean=float(mean[j]),
                hdi_low=float(hdi[j,0]),hdi_high=float(hdi[j,1])))
    return summaries,features

def save_synthetic_data(d,path):
    # Save truth and design before expensive fitting. Deterministic regeneration is checked.
    path=Path(path); yhash=hashlib.sha256(d['y'].tobytes()).hexdigest()
    if path.exists():
        with np.load(path,allow_pickle=False) as saved:
            if hashlib.sha256(saved['y'].tobytes()).hexdigest()!=yhash:
                raise RuntimeError('Saved synthetic outcomes differ from regenerated outcomes')
        return yhash
    tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    with tmp.open('wb') as handle:
        np.savez_compressed(handle,y=d['y'],structural_mean=d['structural_mean'],
            patient_values=d['p'].to_numpy(),patient_columns=np.asarray(d['p'].columns,str),
            fmap=d['fmap'],bmi_sd=np.asarray(d['bmi_sd']),scales=d['scales'],nu=d['nu'])
    os.replace(tmp,path)
    return yhash

def _task_state(folder,tid,stage,**extra):
    atomic_json(dict(config_hash=CONFIG_HASH,task_id=tid,stage=stage,
        updated_utc=datetime.now(timezone.utc).isoformat(),**extra),folder/'state.json')

def _load_checked_trace(path,tid,profile,data_sha):
    receipt=path.with_suffix('.checkpoint.json')
    if receipt.exists():
        expected=json.loads(receipt.read_text())
        if expected['sha256']!=file_hash(path): raise RuntimeError('Trace checksum mismatch: '+str(path))
    obj=az.from_netcdf(path,engine='h5netcdf')
    expected={'config_hash':CONFIG_HASH,'task_id':tid,'profile':profile,'data_sha256':data_sha}
    if any(obj.attrs.get(k)!=v for k,v in expected.items()):
        obj.close();raise RuntimeError('Trace identity mismatch: '+str(path))
    sizes=obj.posterior.sizes
    if sizes.get('chain')!=PROFILES[profile]['chains'] or sizes.get('draw')!=PROFILES[profile]['draws']:
        obj.close();raise RuntimeError('Incomplete trace dimensions: '+str(path))
    return obj

def run_task_v5(g,s,r,kind):
    tid=task_id(g,s,r,kind);folder=OUTPUT_ROOT/'tasks'/tid
    folder.mkdir(parents=True,exist_ok=True);final=folder/'result.json'
    if final.exists():
        old=json.loads(final.read_text())
        if old.get('config_hash')!=CONFIG_HASH or old.get('task_id')!=tid:
            raise RuntimeError('Result identity mismatch')
        print('SKIP',tid,old['status'],flush=True);return old
    t0=time.monotonic();d=simulate(s,r,g);truth=targets(d,kind)
    data_sha=save_synthetic_data(d,folder/'synthetic_data.npz')
    result=dict(task_id=tid,config_hash=CONFIG_HASH,geometry=g,scenario=s.name,
        replicate=r,kind=kind,bmi_sd=d['bmi_sd'],native_to_per5=5/d['bmi_sd'],
        n_patients=len(d['p']),n_features=d['y'].shape[1],data_sha256=data_sha,
        status='pending',estimation_metrics=[],selection_metrics=[],profile_history=[],
        ols_metrics=ols_benchmark(d,kind),estimation_condition='geometry_and_convergence_pass',
        selection_condition='all_prespecified_ROPE_contrast_joint_precision_checks_pass')
    # OLS and truth persist even if a subsequent software or sampling failure stops work.
    atomic_json(result,folder/'task_inputs.json')
    model=None
    for profile in ('publication','publication_long'):
        if profile=='publication_long' and not ALLOW_PUBLICATION_LONG:break
        path=folder/(profile+'.nc');cache=folder/(profile+'_assessment.json')
        if cache.exists() and path.exists():
            assessment=json.loads(cache.read_text())
            if (assessment.get('config_hash')!=CONFIG_HASH or assessment.get('task_id')!=tid
                or assessment.get('profile')!=profile or assessment.get('trace_sha256')!=file_hash(path)):
                raise RuntimeError('Assessment provenance mismatch')
            print('REUSE ASSESSMENT',tid,profile,flush=True)
        else:
            if path.exists():
                _task_state(folder,tid,'loading_saved_trace',profile=profile)
                idata=_load_checked_trace(path,tid,profile,data_sha)
                sample_seconds=None
                timing_path=folder/(profile+'_sampling_time.json')
                if timing_path.exists():sample_seconds=json.loads(timing_path.read_text())['seconds']
            else:
                if model is None:model=build_model(d,kind,s)
                _task_state(folder,tid,'sampling',profile=profile)
                before=time.monotonic()
                idata=sample(model,profile,int(digest([SEED,tid,profile])[:8],16))
                sample_seconds=time.monotonic()-before
                idata.attrs.update(config_hash=CONFIG_HASH,task_id=tid,profile=profile,data_sha256=data_sha)
                _task_state(folder,tid,'saving_trace',profile=profile)
                save_trace(idata,path)
                atomic_json({'seconds':sample_seconds},folder/(profile+'_sampling_time.json'))
            try:
                _task_state(folder,tid,'diagnostics',profile=profile)
                diag=diagnostics(idata)
                assessment=dict(config_hash=CONFIG_HASH,task_id=tid,profile=profile,
                    trace_sha256=file_hash(path),diagnostics=diag,sample_seconds=sample_seconds,
                    estimation_metrics=[],selection_metrics=[],precision={},status='pending')
                if not diag['geometry_pass']:assessment['status']='withheld_geometry'
                elif not diag['convergence_pass']:assessment['status']='withheld_convergence'
                else:
                    effects=extract_effects(idata,kind)
                    metrics,rows=estimation_metrics(effects,truth,d['fmap'])
                    assessment['estimation_metrics']=metrics
                    # No discovery mask is implied by an estimation-only file.
                    atomic_json(dict(config_hash=CONFIG_HASH,task_id=tid,profile=profile,
                        trace_sha256=assessment['trace_sha256'],rows=rows,metrics=metrics,
                        selection_status='not_attached'),folder/(profile+'_estimation.json'))
                    _task_state(folder,tid,'selection_precision',profile=profile)
                    passed,report,masks,joint=assess_all(effects)
                    assessment['precision']=report
                    assessment['status']='completed' if passed else 'withheld_precision'
                    if passed:
                        for name,a in effects.items():
                            assessment['selection_metrics'].append(dict(contrast=name,
                                rule='within_contrast_ROPE_BFDR',**recovery(a,truth[name],masks[name])))
                            if joint:assessment['selection_metrics'].append(dict(contrast=name,
                                rule='joint_categorical_ROPE_BFDR',**recovery(a,truth[name],joint[name])))
                        atomic_json(dict(config_hash=CONFIG_HASH,task_id=tid,profile=profile,
                            trace_sha256=assessment['trace_sha256'],within={k:v.tolist() for k,v in masks.items()},
                            joint={k:v.tolist() for k,v in joint.items()}),folder/(profile+'_selection.json'))
                atomic_json(assessment,cache)
            finally:
                idata.close()
        result['profile_history'].append({k:assessment[k] for k in
            ['profile','status','diagnostics','sample_seconds']})
        result.update(status=assessment['status'],profile=profile,diagnostics=assessment['diagnostics'],
            precision=assessment['precision'],selection_metrics=assessment['selection_metrics'])
        if assessment['estimation_metrics']:
            result['estimation_metrics']=assessment['estimation_metrics']
            result['estimation_profile']=profile
        if assessment['status'] in ('completed','withheld_geometry'):break
    result['estimation_available']=bool(result['estimation_metrics'])
    result['selection_available']=result['status']=='completed'
    result['elapsed_seconds_this_attempt']=time.monotonic()-t0
    atomic_json(result,final);_task_state(folder,tid,'finished',status=result['status'])
    print('FINISHED',tid,result['status'],'estimation:',result['estimation_available'],flush=True)
    return result

def progress_v5(tasks,title):
    from collections import Counter
    counts=Counter();states=Counter();timings={}
    for args in tasks:
        g,s,r,kind=args;tid=task_id(*args);folder=OUTPUT_ROOT/'tasks'/tid
        p=folder/'result.json'
        if p.exists():
            record=json.loads(p.read_text())
            if record.get('config_hash')!=CONFIG_HASH or record.get('task_id')!=tid:
                raise RuntimeError('Progress found incompatible result')
            counts['finished_assessments']+=1;states[record['status']]+=1
            counts['estimation_available']+=int(record['estimation_available'])
            counts['selection_available']+=int(record['selection_available'])
            for h in record['profile_history']:
                if h.get('sample_seconds') is not None:
                    timings.setdefault((g,kind,h['profile']),[]).append(h['sample_seconds'])
        elif any((folder/(p+'.nc')).exists() for p in PROFILES):counts['saved_fit_assessment_unfinished']+=1
        elif (folder/'failure.json').exists():counts['error_without_saved_fit']+=1
        elif (folder/'state.json').exists():counts['started_no_saved_fit']+=1
        else:counts['not_started']+=1
    print('\n'+title,datetime.now(timezone.utc).isoformat())
    print('Experiment:',OUTPUT_ROOT,'\nFinished:',counts['finished_assessments'],'/',len(tasks))
    print(dict(counts));print('Statuses:',dict(states))
    rows=[dict(geometry=g,kind=k,profile=p,n=len(v),median_minutes=float(np.median(v)/60))
          for (g,k,p),v in timings.items()]
    if rows:display(pd.DataFrame(rows));print('Measured sampling times; not a whole-study ETA.')
    return dict(counts)

def run_shard_v5():
    if not RUN_SIMULATION:print('Sampling disabled.');return
    if not SAVE_TRACES:raise ValueError('Checkpoint saving must remain enabled')
    checkpoint_io_preflight(OUTPUT_ROOT,SHARD_ID)
    launched=0
    for args in MY_TASKS:
        tid=task_id(*args);folder=OUTPUT_ROOT/'tasks'/tid
        final=folder/'result.json'
        if final.exists():run_task_v5(*args);continue
        if MAX_NEW_TASKS_PER_LAUNCH is not None and launched>=MAX_NEW_TASKS_PER_LAUNCH:break
        launched+=1
        print(f'Shard {SHARD_ID}, new task {launched}: {tid}',flush=True)
        try:run_task_v5(*args)
        except Exception:
            folder.mkdir(parents=True,exist_ok=True)
            record=dict(config_hash=CONFIG_HASH,task_id=tid,error=traceback.format_exc(),
                created_utc=datetime.now(timezone.utc).isoformat())
            atomic_json(record,folder/('failure_'+uuid.uuid4().hex+'.json'))
            atomic_json(record,folder/'failure.json')
            print(record['error'],flush=True)
            print('STOPPED on unexpected error. No next task launched. Keep this runtime connected.',flush=True)
            raise
        progress_v5(MY_TASKS,f'SHARD {SHARD_ID}')
    print('Launch finished; tasks handled:',launched)
    print('These full-profile tasks count toward the study. Existing results are reused on the next launch.')

def aggregate_v5():
    records=[];missing=[];inputs=[]
    for args in PLANNED_TASKS:
        tid=task_id(*args);folder=OUTPUT_ROOT/'tasks'/tid;p=folder/'result.json'
        if not p.exists():missing.append(tid);continue
        record=json.loads(p.read_text())
        if record.get('config_hash')!=CONFIG_HASH or record.get('task_id')!=tid:raise RuntimeError('Result identity mismatch')
        records.append(record)
    rows=[];status=[]
    for r in records:
        base={k:r[k] for k in ['geometry','scenario','replicate','kind','status']}
        status.append({**base,'estimation_available':r['estimation_available'],
            'selection_available':r['selection_available'],'estimation_profile':r.get('estimation_profile')})
        for source,condition in [('estimation_metrics','geometry_and_convergence'),
                                 ('selection_metrics','all_selection_gates'),('ols_metrics','completed_assessment_tasks')]:
            for metric in r.get(source,[]):rows.append({**base,'condition':condition,**metric})
    frame=pd.DataFrame(rows);summary=[]
    if not frame.empty:
        groupkeys=['geometry','scenario','kind','contrast','rule','condition']
        for key,group in frame.groupby(groupkeys,dropna=False):
            row=dict(zip(groupkeys,key));row.update(n_datasets=len(group),planned_datasets=PLANNED_REPLICATES)
            for metric in ['coverage','active_coverage','practical_null_coverage','bias','mean_absolute_error',
                           'mse','rmse','mean_interval_width','practical_power','classical_fdp','practical_fdp','discoveries']:
                if metric not in group:continue
                v=pd.to_numeric(group[metric],errors='coerce').dropna()
                row[metric+'_mean']=float(v.mean()) if len(v) else None
                row[metric+'_mcse']=float(v.std(ddof=1)/np.sqrt(len(v))) if len(v)>1 else None
                row[metric+'_n']=len(v)
            summary.append(row)
    destination=OUTPUT_ROOT/'reports'/f'shard_{SHARD_ID}'
    destination.mkdir(parents=True,exist_ok=True)
    table=pd.DataFrame(summary);frame.to_csv(destination/'replicate_metrics.csv',index=False)
    table.to_csv(destination/'performance_summary.csv',index=False)
    pd.DataFrame(status).to_csv(destination/'task_statuses.csv',index=False)
    atomic_json(dict(config_hash=CONFIG_HASH,planned_tasks=len(PLANNED_TASKS),finished=len(records),
        missing=missing,summary=summary,
        note='Monte Carlo SEs use datasets as replicates, not pathways. Withheld selections are unavailable, never zero discoveries. '
             'Bayes estimation is conditional on geometry/convergence; selection on all precision gates. '
             'Mean dataset RMSE is reported separately from mean MSE. No matched-estimand power-superiority claim.'),
        destination/'summary.json')
    display(table);print('Saved reports:',destination)
    export_simulation_figures(records,table,destination)
    return table


In [ ]:
# 8. Experiment identity, balanced ownership and operational prefix
assert SUITE=='focused'
assert 1 <= REPLICATES_TO_RUN <= PLANNED_REPLICATES
assert MAX_NEW_TASKS_PER_LAUNCH is None or MAX_NEW_TASKS_PER_LAUNCH>=1
SCENARIOS=focused_scenarios()
FUNCTIONS=[patient_frame,design,residual_basis,simulate,targets,build_model,extract_effects,sample,
    diagnostics,assess_all,recovery,ols_benchmark,resolve_geometry,_az_da_stat,
    effective_sample_boundary_guard,rope_indicator_metrics,deterministic_bfdr,bounded_series_mc,
    selection_sets,assess_rope_bfdr,focused_scenarios,design_cases,resolve_geometry_v5,task_schedule,estimation_metrics,
    run_task_v5,_load_checked_trace,save_synthetic_data,aggregate_v5,save_trace,_netcdf_copy,export_simulation_figures]
CODE_HASH=hashlib.sha256('\n'.join(inspect.getsource(f) for f in FUNCTIONS).encode()).hexdigest()
CONTRACT=dict(version=VERSION,code_hash=CODE_HASH,io_version=IO_REPAIR_VERSION,suite=SUITE,seed=SEED,
    planned_replicates=PLANNED_REPLICATES,geometry=GEOMETRY,scenarios=[asdict(s) for s in SCENARIOS],
    priors=PRIORS,profiles=PROFILES,allow_long=ALLOW_PUBLICATION_LONG,
    rope_grid=ROPE_GRID,primary_rope=PRIMARY_ROPE,q=Q_LEVEL,
    precision=dict(indicator_ess=ROPE_INDICATOR_ESS_MIN,indicator_mcse=ROPE_LOCAL_NULL_MCSE_MAX,
        bfdr_mcse=BFDR_MCSE_MAX,alpha=MC_ALPHA),
    versions={p:importlib.metadata.version(p) for p in
        ['pymc','pytensor','nutpie','arviz','numpy','pandas','scipy','xarray','h5netcdf','h5py']},
    selection_policy='v4_all_rope_and_scope_precision',
    estimation_policy='latest_converged_profile_independent_of_selection',
    ownership='rotating_pair_index_plus_replicate_mod_n_shards',save_traces=SAVE_TRACES)
CONFIG_HASH=digest(CONTRACT);OUTPUT_ROOT=OUTPUT_BASE/('focused_'+CONFIG_HASH[:16])
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
manifest=dict(contract=CONTRACT,config_hash=CONFIG_HASH,n_shards=N_SHARDS)
path=OUTPUT_ROOT/'experiment.json'
if path.exists():
    if json.loads(path.read_text())!=json.loads(json.dumps(manifest,default=str)):
        raise RuntimeError('Experiment/shard contract mismatch. Do not relabel existing output.')
else:atomic_json(manifest,path)
PLANNED_TASKS,_=task_schedule(GEOMETRIES,SCENARIOS,PLANNED_REPLICATES,N_SHARDS,SHARD_ID)
TASKS,MY_TASKS=task_schedule(GEOMETRIES,SCENARIOS,REPLICATES_TO_RUN,N_SHARDS,SHARD_ID)
print('Experiment:',OUTPUT_ROOT)
print('Planned initial fits:',len(PLANNED_TASKS),'potential extra long fits:',len(PLANNED_TASKS) if ALLOW_PUBLICATION_LONG else 0)
print('Active prefix:',len(TASKS),'This shard:',len(MY_TASKS),'First-launch cap:',MAX_NEW_TASKS_PER_LAUNCH)
display(pd.DataFrame([dict(geometry=g,celltypes=len(v['counts']),pathways=sum(v['counts']),
    source='audited pre-fit inventory' if v['source'] else 'GENERIC: fixed layout shown in contract') for g,v in GEOMETRY.items()]))
display(pd.DataFrame([dict(repetitions=r,MCSE_near_5_or_95_percent=(.05*.95/r)**.5,
    worst_case_MCSE=(.25/r)**.5) for r in [10,25,50,100,500,1000]]))
display(pd.DataFrame([dict(geometry=g,scenario=s.name,patients=sum(s.counts),group_counts=s.counts,
    effect=s.effect,shape=s.shape,pathways=sum(GEOMETRY[g]['counts'])) for g,s in design_cases(GEOMETRIES,SCENARIOS)]))
print('MCSE table illustrates a binary dataset-level event; actual summaries use independent dataset variability.')
checkpoint_io_preflight(OUTPUT_ROOT,SHARD_ID)
progress_v5(TASKS,'ALL ACTIVE SHARDS')


In [ ]:
# 7. Structural checks: no MCMC, no posterior sampling
def structural_tests():
    g=GEOMETRIES[0]; s=Scenario('test'); d=simulate(s,0,g)
    assert tuple(d['p'].group.value_counts().sort_index())==(52,57,18)
    for kind in ('continuous','categorical'):
        x,_,_=design(d['p'],kind); b=residual_basis(x)
        assert b.shape[1]==127-(4 if kind=='continuous' else 5)
        m=build_model(d,kind,s)
        assert np.isfinite(m.compile_logp()(m.initial_point()))
    a=simulate(replace(s,bmi_prior_multiplier=.5),0,g)
    z=simulate(replace(s,bmi_prior_multiplier=2),0,g)
    np.testing.assert_array_equal(a['y'],z['y'])
    t=targets(d,'categorical')
    np.testing.assert_allclose(t['obese_vs_overweight'],t['obese_vs_normal']-t['overweight_vs_normal'])
    dn=simulate(replace(s,shape='null',effect=0),0,g)
    for kind in ('continuous','categorical'):
        for truth in targets(dn,kind).values(): np.testing.assert_allclose(truth,0,atol=1e-10)
    print('PASS: group allocation, basis rank/normalization/orthogonality, initial logp, paired prior data, contrast algebra, null targets.')
structural_tests()

In [ ]:
# 9. Launch full-profile tasks
run_shard_v5()


In [ ]:
# 10. Current progress and aggregate reporting
progress_v5(TASKS,'ALL ACTIVE SHARDS')
if AGGREGATE:
    simulation_summary=aggregate_v5()
else:
    print('Aggregation disabled.')


In [ ]:
# 12. Download the report bundle; complete posterior traces remain in Drive
if DOWNLOAD_REPORTS:
    import shutil
    report_dir = OUTPUT_ROOT/'reports'/f'shard_{SHARD_ID}'
    if not report_dir.is_dir():
        raise FileNotFoundError('Run aggregate reporting first')
    archive = shutil.make_archive(str(OUTPUT_ROOT/f'reports_shard_{SHARD_ID}'),'zip',report_dir)
    print('Saved report archive:',archive)
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
